# RNN（Recurrent Neural Network）を理解する

このノートブックでは、RNN を **5つのステップ** で段階的に学びます。

| Step | 内容 | 目的 |
|------|------|------|
| 1 | RNNセルの中身を手計算で理解 | 数式とデータの流れを体感 |
| 2 | NumPy でRNNセルをスクラッチ実装 | 順伝播の仕組みを手で書く |
| 3 | PyTorch の `nn.RNN` で正弦波予測 | 実用的な使い方を学ぶ |
| 4 | 学習結果の可視化と考察 | RNN の強みと限界を理解 |
| 5 | Scheduled Sampling で改善 | Teacher Forcing の問題を解決 |

---
## Step 1: RNN セルの中身を理解する

### 通常のニューラルネットワーク（MLP）との違い

MLP では入力 → 出力が **1回で完結** しますが、RNN は **前の時刻の情報を引き継いで** 次の計算に使います。

```
MLP:   入力 x → [W] → 出力 y    （毎回独立）

RNN:   x₀ → [W, U] → h₀ ─┐
       x₁ → [W, U] → h₁ ─┤  ← 前の隠れ状態が次に渡される
       x₂ → [W, U] → h₂ ─┘
```

### RNN セルの数式

各時刻 t での計算は **たった1行** です：

$$h_t = \tanh(W_{ih} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

- $x_t$: 時刻 t の入力ベクトル
- $h_{t-1}$: 前の時刻の隠れ状態（**記憶**）
- $W_{ih}$: 入力→隠れ層の重み
- $W_{hh}$: 隠れ層→隠れ層の重み（**ここがRNNの核心**）
- $\tanh$: 活性化関数（-1〜1 に収める）

つまり「**新しい入力** + **前の記憶**」を合成して、新しい記憶を作っています。

### 手計算で追いかけてみよう

具体的な数値で RNN セルの動きを確認します。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# --- 設定 ---
# 入力の次元: 1（スカラー値を1つ受け取る）
# 隠れ状態の次元: 2（2つの値で「記憶」を保持）
input_size = 1
hidden_size = 2

# 重みを手動で設定（理解しやすい小さな値）
W_ih = np.array([[0.5], [0.3]])    # shape: (hidden_size, input_size) = (2, 1)
W_hh = np.array([[0.1, 0.2],       # shape: (hidden_size, hidden_size) = (2, 2)
                 [0.3, 0.4]])
b = np.array([0.0, 0.0])           # バイアス（今回は0）

# 初期隠れ状態（最初は記憶がないので0）
h = np.array([0.0, 0.0])

# 入力系列: [1.0, 0.5, -0.3] の3時刻分
inputs = [1.0, 0.5, -0.3]

print("=== RNN セルを手動で実行 ===")
print(f"W_ih (入力→隠れ): {W_ih.flatten()}")
print(f"W_hh (隠れ→隠れ): \n{W_hh}")
print(f"初期隠れ状態 h₀: {h}")
print()

for t, x_t in enumerate(inputs):
    x_t = np.array([x_t])  # スカラーをベクトルに
    
    # RNN の核心: h_t = tanh(W_ih @ x_t + W_hh @ h_{t-1} + b)
    raw = W_ih @ x_t + W_hh @ h + b
    h_new = np.tanh(raw)
    
    print(f"--- 時刻 t={t} ---")
    print(f"  入力 x_{t}: {x_t}")
    print(f"  前の隠れ状態 h_{t}: {h}")
    print(f"  W_ih @ x  = {(W_ih @ x_t)}")
    print(f"  W_hh @ h  = {(W_hh @ h)}")
    print(f"  合計 (tanh前) = {raw}")
    print(f"  h_{t+1} = tanh(合計) = {h_new}")
    print()
    
    h = h_new  # 隠れ状態を更新（← これが "再帰" の部分）

**ポイント**: 時刻が進むにつれて、`W_hh @ h` の部分が **過去の入力の影響を含んだ値** になっていることを確認してください。

これが RNN が「時系列の記憶」を持つ仕組みです。

---
## Step 2: NumPy で RNN をスクラッチ実装

Step 1 の手計算をクラスにまとめます。
PyTorch を使わず、NumPy だけで RNN の **順伝播（forward pass）** を実装します。

In [ ]:
class SimpleRNN:
    """NumPy で実装するシンプルな RNN（順伝播のみ）"""
    
    def __init__(self, input_size, hidden_size):
        # Xavier初期化（重みが大きすぎず小さすぎない初期値）
        scale_ih = np.sqrt(1.0 / input_size)
        scale_hh = np.sqrt(1.0 / hidden_size)
        
        self.W_ih = np.random.randn(hidden_size, input_size) * scale_ih
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.b = np.zeros(hidden_size)
        self.hidden_size = hidden_size
    
    def forward(self, x_sequence, h_init=None):
        """
        順伝播: 入力系列を受け取り、各時刻の隠れ状態を返す
        
        Args:
            x_sequence: shape (seq_len, input_size) の入力系列
            h_init: 初期隠れ状態。None なら0ベクトル
        
        Returns:
            hiddens: shape (seq_len, hidden_size) の隠れ状態の系列
        """
        seq_len = x_sequence.shape[0]
        
        if h_init is None:
            h = np.zeros(self.hidden_size)
        else:
            h = h_init
        
        hiddens = []
        for t in range(seq_len):
            x_t = x_sequence[t]  # この時刻の入力
            h = np.tanh(self.W_ih @ x_t + self.W_hh @ h + self.b)
            hiddens.append(h)
        
        return np.array(hiddens)


# 動作確認
rnn = SimpleRNN(input_size=1, hidden_size=4)

# 正弦波を入力として使う
t = np.linspace(0, 4 * np.pi, 50)
x_seq = np.sin(t).reshape(-1, 1)  # shape: (50, 1)

hiddens = rnn.forward(x_seq)
print(f"入力系列の形状: {x_seq.shape}")
print(f"隠れ状態の形状: {hiddens.shape}")
print(f"\n最初の5時刻の隠れ状態:")
print(hiddens[:5])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# 入力（正弦波）
axes[0].plot(t, x_seq, 'b-', linewidth=2)
axes[0].set_ylabel('Input (sin)')
axes[0].set_title('RNN の入出力を可視化')
axes[0].grid(True, alpha=0.3)

# 各隠れユニットの出力
for i in range(hiddens.shape[1]):
    axes[1].plot(t, hiddens[:, i], label=f'h[{i}]', linewidth=1.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Hidden state')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("各隠れユニットが入力の正弦波に対して異なる反応をしていることを確認してください。")
print("これが RNN の『特徴抽出』です。重みが違うので、同じ入力でも異なるパターンを捉えます。")

---
## Step 3: PyTorch で正弦波予測

ここからが本題です。PyTorch の `nn.RNN` を使って、**正弦波の次の値を予測** するモデルを作ります。

### なぜ正弦波？
- 規則的なパターンなので、RNN が学習できたかどうかが **一目でわかる**
- 音声波形も突き詰めると正弦波の合成なので、音声処理への第一歩になる

### タスク設計
```
入力:  [sin(t₀), sin(t₁), ..., sin(t₁₉)]    ← 20時刻分の値
正解:  [sin(t₁), sin(t₂), ..., sin(t₂₀)]    ← 1時刻ずらした値
```

つまり「過去20点から、各時刻の次の値を当てる」タスクです。

In [ ]:
import torch
import torch.nn as nn

# --- デバイス設定 ---
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"使用デバイス: {device}")

In [ ]:
# --- データセット作成 ---

def create_sine_dataset(n_samples=1000, seq_len=20, freq_range=(0.5, 2.0)):
    """
    正弦波の予測データセットを作成
    
    異なる周波数の正弦波をランダムに生成し、
    「seq_len 個の値から次の値を予測する」タスクにする
    """
    X_list = []
    y_list = []
    
    for _ in range(n_samples):
        # ランダムな周波数と位相
        freq = np.random.uniform(*freq_range)
        phase = np.random.uniform(0, 2 * np.pi)
        
        # seq_len + 1 個の連続した値を生成
        start = np.random.uniform(0, 10)
        t = np.linspace(start, start + 2 * np.pi, seq_len + 1)
        values = np.sin(freq * t + phase)
        
        # 入力: 最初の seq_len 個、ターゲット: 1つずらした seq_len 個
        X_list.append(values[:-1])
        y_list.append(values[1:])
    
    X = torch.FloatTensor(np.array(X_list)).unsqueeze(-1)  # (n_samples, seq_len, 1)
    y = torch.FloatTensor(np.array(y_list)).unsqueeze(-1)  # (n_samples, seq_len, 1)
    
    return X, y


# データ作成
X_train, y_train = create_sine_dataset(n_samples=2000, seq_len=20)
X_test, y_test = create_sine_dataset(n_samples=200, seq_len=20)

print(f"訓練データ: X={X_train.shape}, y={y_train.shape}")
print(f"テストデータ: X={X_test.shape}, y={y_test.shape}")
print(f"\nX の意味: (サンプル数, 系列長, 入力次元)")
print(f"各サンプルは {X_train.shape[1]} 時刻分の正弦波の値")

In [ ]:
# データの中身を確認
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    x_sample = X_train[i].squeeze().numpy()
    y_sample = y_train[i].squeeze().numpy()
    
    ax.plot(range(20), x_sample, 'bo-', markersize=4, label='入力 (t)')
    ax.plot(range(20), y_sample, 'rx-', markersize=4, label='正解 (t+1)')
    ax.set_title(f'サンプル {i}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('入力と正解の関係（入力を1時刻ずらしたものが正解）', fontsize=12)
plt.tight_layout()
plt.show()

### RNN モデルの定義

PyTorch の `nn.RNN` は、Step 2 で自作した `SimpleRNN` と同じことを、
GPU対応・自動微分付きでやってくれます。

```
入力 (batch, seq_len, 1)
  ↓
nn.RNN(input_size=1, hidden_size=32)  ← RNN層
  ↓
隠れ状態 (batch, seq_len, 32)
  ↓
nn.Linear(32, 1)  ← 出力層（隠れ状態→予測値に変換）
  ↓
出力 (batch, seq_len, 1)
```

In [ ]:
class SineRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # RNN 層
        # batch_first=True: 入力を (batch, seq_len, features) の順にする
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        
        # 出力層: 隠れ状態(32次元) → 予測値(1次元)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x, h_0=None):
        """
        Args:
            x: (batch, seq_len, input_size) の入力
            h_0: 初期隠れ状態。None なら 0 で初期化
        
        Returns:
            output: (batch, seq_len, 1) の予測値
        """
        # h_0 の shape: (num_layers, batch, hidden_size)
        if h_0 is None:
            h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # RNN に通す
        # rnn_out: (batch, seq_len, hidden_size) ... 全時刻の隠れ状態
        # h_n:     (num_layers, batch, hidden_size) ... 最後の隠れ状態
        rnn_out, h_n = self.rnn(x, h_0)
        
        # 全時刻の隠れ状態を出力層に通す
        output = self.fc(rnn_out)
        
        return output


model = SineRNN(input_size=1, hidden_size=32).to(device)
print(model)
print(f"\n総パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

### 学習ループ

MLP/CNN と同じ流れです：
1. 順伝播（forward）
2. 損失計算（MSE: 予測値と正解の二乗誤差）
3. 逆伝播（backward）
4. パラメータ更新（optimizer.step）

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# ハイパーパラメータ
EPOCHS = 50
BATCH_SIZE = 64
LR = 0.005

# DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 損失関数とオプティマイザ
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"エポック数: {EPOCHS}")
print(f"バッチサイズ: {BATCH_SIZE}")
print(f"学習率: {LR}")
print(f"バッチ数/エポック: {len(train_loader)}")

In [ ]:
# --- 学習 ---
train_losses = []
test_losses = []

for epoch in range(EPOCHS):
    # --- 訓練 ---
    model.train()
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        
        # 勾配クリッピング: RNN は勾配爆発しやすいので制限する
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # --- 検証 ---
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            test_loss += criterion(output, y_batch).item()
    
    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}]  "
              f"Train Loss: {train_loss:.6f}  "
              f"Test Loss: {test_loss:.6f}")

print("\n学習完了!")

---
## Step 4: 結果の可視化と考察

In [ ]:
# --- 損失曲線 ---
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('学習曲線')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

In [ ]:
# --- 予測結果の可視化 ---
model.eval()

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        x_sample = X_test[i].unsqueeze(0).to(device)  # (1, seq_len, 1)
        y_true = y_test[i].squeeze().numpy()
        y_pred = model(x_sample).squeeze().cpu().numpy()
        
        time_steps = range(len(y_true))
        
        ax.plot(time_steps, X_test[i].squeeze().numpy(), 'b.-', 
                alpha=0.5, label='入力', markersize=4)
        ax.plot(time_steps, y_true, 'g.-', 
                label='正解 (次の値)', markersize=4)
        ax.plot(time_steps, y_pred, 'r.--', 
                label='RNN予測', markersize=4)
        ax.set_title(f'テストサンプル {i}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('RNN による正弦波の次の値予測', fontsize=14)
plt.tight_layout()
plt.show()

### 自由生成: RNN に正弦波を「続き」を描かせる

最初の数点だけ与えて、あとは RNN の予測を次の入力として使い、
自分自身で波形を生成させます（**自己回帰生成**）。

```
初期入力: [sin(0), sin(1), ..., sin(4)]  ← 最初の5点だけ本物
     ↓
RNN予測: sin(5)' を予測
     ↓
次の入力に使う: [..., sin(4), sin(5)']  ← 予測値を入力に
     ↓
RNN予測: sin(6)' を予測 ...
```

これは音声合成やテキスト生成と同じ原理です。

In [ ]:
def autoregressive_generate(model, seed_values, n_generate=100):
    """
    seed_values（初期値）を起点に、RNNに自己回帰で波形を生成させる
    """
    model.eval()
    
    # seed をテンソルに変換
    generated = list(seed_values)
    input_seq = torch.FloatTensor(seed_values).reshape(1, -1, 1).to(device)
    
    with torch.no_grad():
        # まず seed 全体を RNN に通して隠れ状態を作る
        h = torch.zeros(1, 1, model.hidden_size).to(device)
        output, h = model.rnn(input_seq, h)
        last_pred = model.fc(output[:, -1:, :])  # 最後の時刻の予測
        
        generated.append(last_pred.item())
        
        # 自己回帰: 予測を次の入力にして繰り返す
        for _ in range(n_generate - 1):
            next_input = last_pred  # (1, 1, 1)
            output, h = model.rnn(next_input, h)
            last_pred = model.fc(output)
            generated.append(last_pred.item())
    
    return np.array(generated)


# --- 自由生成の実行 ---
seed_len = 10
total_len = 200

# 正弦波の最初の seed_len 点を与える
t_true = np.linspace(0, 8 * np.pi, total_len)
true_wave = np.sin(t_true)
seed = true_wave[:seed_len]

generated = autoregressive_generate(model, seed, n_generate=total_len - seed_len)

# --- 可視化 ---
plt.figure(figsize=(14, 5))

# seed 部分（灰色）: 与えた初期値
plt.plot(t_true[:seed_len], generated[:seed_len], 'o-',
         color='gray', linewidth=2, markersize=5, label=f'seed (最初の{seed_len}点)')

# RNN 生成部分（赤）: seed 以降の自己回帰生成
plt.plot(t_true[seed_len:len(generated)], generated[seed_len:], 'r-',
         linewidth=1.5, label='RNN generated')

# 正解の正弦波（青）
plt.plot(t_true, true_wave, 'b--', alpha=0.4, linewidth=2, label='Ground truth (sin)')

# seed と生成の境界線
plt.axvline(x=t_true[seed_len], color='gray', linestyle=':', alpha=0.5)

plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Autoregressive Generation by RNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("灰色の点が seed（本物のsin波から与えた初期値）、赤い線が RNN の自己回帰生成です。")
print("青い破線（正解）と赤い線が近いほど、RNN が正弦波のパターンを学習できています。")
print("時間が経つにつれて誤差が蓄積し、波形が崩れることがあります。")
print("-> これが RNN の限界であり、LSTM が必要になる理由の一つです。")

---
## Step 5: Scheduled Sampling で生成品質を改善する

### Teacher Forcing の問題（おさらい）

```
【学習時】正解を入力に使う     →  常に "綺麗な入力" で練習
【生成時】予測を入力に使う     →  ズレた入力に対応できない
```

### Scheduled Sampling のアイデア

学習時に「たまに自分の予測を次の入力に使う」ことで、ズレた入力にも慣れさせます。

```
エポック初期: ほぼ正解を使う（Teacher Forcing率 高い）
  入力: 正解 → 正解 → 正解 → 予測 → 正解 → ...

エポック後半: 予測を使う割合を増やす
  入力: 予測 → 正解 → 予測 → 予測 → 予測 → ...
```

### 学習戦略: 2段階で進める

いきなり予測を混ぜると、予測がデタラメなうちにそれで練習してしまい崩壊します。
そこで **Phase 1 で基礎を固めてから、Phase 2 で徐々に予測を混ぜる** 方式にします。

```
Phase 1 (epoch  0-29): Teacher Forcing のみ → まず正確な予測を学ぶ
Phase 2 (epoch 30-99): TF率 1.0 → 0.5 に緩やかに減衰 → ズレへの耐性をつける
```

In [ ]:
# --- Scheduled Sampling 付き学習（2段階方式） ---

model_ss = SineRNN(input_size=1, hidden_size=32).to(device)
optimizer_ss = torch.optim.Adam(model_ss.parameters(), lr=LR)

PHASE1_EPOCHS = 30   # Teacher Forcing のみで基礎を固める
PHASE2_EPOCHS = 70   # Scheduled Sampling で耐性をつける
TOTAL_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS
TF_MIN = 0.5         # TF率の下限（0にすると崩壊しやすい）

train_losses_ss = []
test_losses_ss = []

for epoch in range(TOTAL_EPOCHS):
    model_ss.train()
    epoch_loss = 0.0
    
    # Phase 1: TF率 = 1.0（通常の Teacher Forcing）
    # Phase 2: TF率 = 1.0 → TF_MIN に線形減衰
    if epoch < PHASE1_EPOCHS:
        tf_ratio = 1.0
    else:
        progress = (epoch - PHASE1_EPOCHS) / PHASE2_EPOCHS
        tf_ratio = 1.0 - (1.0 - TF_MIN) * progress
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        batch_size = X_batch.size(0)
        seq_len = X_batch.size(1)
        
        optimizer_ss.zero_grad()
        
        if tf_ratio >= 1.0:
            # Phase 1: 通常の forward（高速）
            output = model_ss(X_batch)
        else:
            # Phase 2: 1時刻ずつループして予測を混ぜる
            h = torch.zeros(1, batch_size, model_ss.hidden_size).to(device)
            outputs = []
            current_input = X_batch[:, 0:1, :]
            
            for t_step in range(seq_len):
                rnn_out, h = model_ss.rnn(current_input, h)
                pred = model_ss.fc(rnn_out)
                outputs.append(pred)
                
                if t_step < seq_len - 1:
                    if np.random.random() < tf_ratio:
                        current_input = X_batch[:, t_step+1:t_step+2, :]
                    else:
                        current_input = pred.detach()
            
            output = torch.cat(outputs, dim=1)
        
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ss.parameters(), max_norm=1.0)
        optimizer_ss.step()
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader)
    train_losses_ss.append(train_loss)
    
    # --- 検証 ---
    model_ss.eval()
    test_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model_ss(X_batch)
            test_loss += criterion(output, y_batch).item()
    test_loss /= len(test_loader)
    test_losses_ss.append(test_loss)
    
    if (epoch + 1) % 20 == 0:
        phase = "Phase1(TF only)" if epoch < PHASE1_EPOCHS else "Phase2(SS)"
        print(f"Epoch [{epoch+1:3d}/{TOTAL_EPOCHS}]  "
              f"Train: {train_loss:.6f}  Test: {test_loss:.6f}  "
              f"TF: {tf_ratio:.2f}  [{phase}]")

print("\nScheduled Sampling 学習完了!")

In [ ]:
# --- 両モデルの自己回帰生成を比較 ---

seed_len = 10
total_len = 200

t_true = np.linspace(0, 8 * np.pi, total_len)
true_wave = np.sin(t_true)
seed = true_wave[:seed_len]

gen_tf = autoregressive_generate(model, seed, n_generate=total_len - seed_len)
gen_ss = autoregressive_generate(model_ss, seed, n_generate=total_len - seed_len)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Teacher Forcing のみ ---
ax = axes[0]
ax.plot(t_true, true_wave, 'b--', alpha=0.4, linewidth=2, label='Ground truth')
ax.plot(t_true[:seed_len], gen_tf[:seed_len], 'o-',
        color='gray', linewidth=2, markersize=4, label='seed')
ax.plot(t_true[seed_len:len(gen_tf)], gen_tf[seed_len:], 'r-',
        linewidth=1.5, label='Generated')
ax.axvline(x=t_true[seed_len], color='gray', linestyle=':', alpha=0.5)
ax.set_ylabel('Value')
ax.set_title('Teacher Forcing only (Step 3-4)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# --- Scheduled Sampling ---
ax = axes[1]
ax.plot(t_true, true_wave, 'b--', alpha=0.4, linewidth=2, label='Ground truth')
ax.plot(t_true[:seed_len], gen_ss[:seed_len], 'o-',
        color='gray', linewidth=2, markersize=4, label='seed')
ax.plot(t_true[seed_len:len(gen_ss)], gen_ss[seed_len:], 'r-',
        linewidth=1.5, label='Generated')
ax.axvline(x=t_true[seed_len], color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('Scheduled Sampling (Step 5)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("上: Teacher Forcing のみ（学習時は常に正解を入力）")
print("下: Scheduled Sampling（学習時に予測値も入力に使って練習）")
print()
print("Scheduled Sampling の方が振幅や周波数を維持できていれば、")
print("「ズレた入力への耐性」が学習できている証拠です。")

---
## まとめ

### 今回学んだこと

| 項目 | 内容 |
|------|------|
| RNN の核心 | $h_t = \tanh(W_{ih} x_t + W_{hh} h_{t-1} + b)$ |
| 隠れ状態 | 過去の入力の情報を圧縮した「記憶」ベクトル |
| PyTorch での使い方 | `nn.RNN` → 全時刻の隠れ状態を取得 → `nn.Linear` で出力 |
| 勾配クリッピング | RNN は勾配爆発しやすいので `clip_grad_norm_` で制限 |
| 自己回帰生成 | 予測を次の入力にして連鎖的に生成（音声合成と同じ原理） |
| Teacher Forcing | 学習時に正解を入力に使う手法。生成時とのギャップが問題 |
| Scheduled Sampling | 学習中に予測値も入力に混ぜて、ギャップを緩和する手法 |

### RNN の限界（次のステップへの動機）

- **勾配消失**: 長い系列では最初の方の情報が失われる
- **長期依存の学習が苦手**: 100時刻前の情報を使うのが難しい
- → **LSTM** はゲート機構でこれを解決する（次回）

### 次のステップ

1. **LSTM を実装** して同じタスクで比較
2. **音声データ（MFCC特徴量）** を入力にして、音声分類に挑戦